<a href="https://colab.research.google.com/github/MrRichar02/Proyecto-Final-Modelos-2-G04/blob/main/desarrollo-proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importaciones generales

Asegurese de instalar las dependencias definidas para el proyecto en el `requirements.txt` o el `pyproject.toml`.

Si esta en colab solo necesita instalar la siguiente dependencia

In [2]:
!pip install ucimlrepo

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

## Obtención de dataset siguiendo la documentación de UC Irvine

In [46]:

# fetch dataset
online_shoppers_purchasing_intention_dataset = fetch_ucirepo(id=468)

# data (as pandas dataframes)
X = online_shoppers_purchasing_intention_dataset.data.features
y = online_shoppers_purchasing_intention_dataset.data.targets.copy()

online_shoppers_purchasing_intention_dataset['variables']

,name,role,type,demographic,description,units,missing_values
0,Administrative,Feature,Integer,None,None,None,no
1,Administrative_Duration,Feature,Integer,None,None,None,no
2,Informational,Feature,Integer,None,None,None,no
3,Informational_Duration,Feature,Integer,None,None,None,no
4,ProductRelated,Feature,Integer,None,None,None,no
5,ProductRelated_Duration,Feature,Continuous,None,None,None,no
6,BounceRates,Feature,Continuous,None,None,None,no
7,ExitRates,Feature,Continuous,None,None,None,no
8,PageValues,Feature,Integer,None,None,None,no
9,SpecialDay,Feature,Integer,None,None,None,no


In [5]:
X['Informational_Duration'].dtype

dtype('float64')

## Análisis variables

De acuerdo a la descripción del dataset disponible en UC Irvine, se concluye que las siguientes variables son categóricas.

- Revenue
- Weekend
- VisitorType
- TrafficType
- Region
- Browser
- OperatingSystems
- Month

In [6]:
vars_cat = ["Weekend","VisitorType","TrafficType","Region","Browser","OperatingSystems","Month"]
print("Revenue: ",np.unique(y["Revenue"]))
for i in vars_cat:
    print(i+": ", np.unique(X[i]))

Revenue:  [False  True]
Weekend:  [False  True]
VisitorType:  ['New_Visitor' 'Other' 'Returning_Visitor']
TrafficType:  [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
Region:  [1 2 3 4 5 6 7 8 9]
Browser:  [ 1  2  3  4  5  6  7  8  9 10 11 12 13]
OperatingSystems:  [1 2 3 4 5 6 7 8]
Month:  ['Aug' 'Dec' 'Feb' 'Jul' 'June' 'Mar' 'May' 'Nov' 'Oct' 'Sep']


## Descripción de las variables numéricas

In [7]:
num_cols = ['Administrative','Administrative_Duration','Informational',
            'Informational_Duration','ProductRelated','ProductRelated_Duration',
            'BounceRates','ExitRates','PageValues','SpecialDay']

In [8]:
for i in num_cols:
    print(i+": ", X[i].dtype)

Administrative:  int64
Administrative_Duration:  float64
Informational:  int64
Informational_Duration:  float64
ProductRelated:  int64
ProductRelated_Duration:  float64
BounceRates:  float64
ExitRates:  float64
PageValues:  float64
SpecialDay:  float64


In [9]:
X[num_cols].describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000


## Preparación encoding variables

### One-hot Encoding

In [47]:
X = pd.get_dummies(X, columns=['OperatingSystems'], prefix="OperatingSytem_", dtype=int)
X = pd.get_dummies(X, columns=['Browser'], prefix="Browser_", dtype=int)
X = pd.get_dummies(X, columns=['Region'], prefix="Region_", dtype=int)
X = pd.get_dummies(X, columns=['TrafficType'], prefix="TrafficType_", dtype=int)
X = pd.get_dummies(X, columns=['VisitorType'], prefix="VisitorType_", dtype=int)

### Label Encoding

In [48]:
X['Month'] = X['Month'].map({
    'Feb': 1,
    'Mar': 2,
    'May': 3,
    'June': 4,
    'Jul': 5,
    'Aug': 6,
    'Sep': 7,
    'Oct': 8,
    'Nov': 9,
    'Dec': 10
})

### Convertir valores tipos Boolean a int(0 y 1)

In [49]:
X['Weekend'] = X['Weekend'].astype(int)

In [50]:
y['Revenue'] = y['Revenue'].astype(int)

### Nuevos shapes

In [14]:
X.shape, y.shape


((12330, 65), (12330, 1))

### Normalización de características reales

In [51]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X[num_cols] = scaler.fit_transform(X[num_cols])

In [24]:
X[num_cols].describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay
count,1.233000e+04,1.233000e+04,1.233000e+04,1.233000e+04,1.233000e+04,1.233000e+04,1.233000e+04,1.233000e+04,1.233000e+04,1.233000e+04
mean,-2.996612e-17,6.281359e-17,-4.610172e-18,-2.535595e-17,4.610172e-17,-8.298309e-17,-6.454241e-17,3.688137e-17,1.060340e-16,-5.532206e-17
std,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00,1.000041e+00
min,-6.969930e-01,-4.571914e-01,-3.964779e-01,-2.449305e-01,-7.134884e-01,-6.243475e-01,-4.576830e-01,-8.863706e-01,-3.171778e-01,-3.088214e-01
25%,-6.969930e-01,-4.571914e-01,-3.964779e-01,-2.449305e-01,-5.560920e-01,-5.281214e-01,-4.576830e-01,-5.923930e-01,-3.171778e-01,-3.088214e-01
50%,-3.959377e-01,-4.147639e-01,-3.964779e-01,-2.449305e-01,-3.087548e-01,-3.113566e-01,-3.934903e-01,-3.686913e-01,-3.171778e-01,-3.088214e-01
75%,5.072280e-01,7.035981e-02,-3.964779e-01,-2.449305e-01,1.409492e-01,1.407881e-01,-1.109348e-01,1.425510e-01,-3.171778e-01,-3.088214e-01
max,7.431499e+00,1.876956e+01,1.849960e+01,1.786868e+01,1.513858e+01,3.280678e+01,3.667189e+00,3.229316e+00,1.916634e+01,4.718598e+00


## Separación de test y train y aplicación de SMOTE

### Dividimos el dataset en test y train, asignando un 20% para el test

In [52]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Aplicación de SMOTE para generar muestras artificiales de la clase minoritaria

#### Se obtiene la relación entre la clase positiva y la clase negativa

In [53]:
clase_negativo, clase_positivo = y_train.value_counts()
(clase_positivo)/(clase_negativo)

0.1830175101942912

#### Primer porcentaje: Se aumenta la proporción de un 18% a un 30%

In [54]:
# Aplicar SMOTE SOLO al train
smote = SMOTE(random_state=42, sampling_strategy=0.3)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train,
    y_train
)

In [55]:
print(y_train.value_counts())
print(y_train_resampled.value_counts())

Revenue
0          8338
1          1526
Name: count, dtype: int64
Revenue
0          8338
1          2501
Name: count, dtype: int64


#### Segundo porcentaje: Se aumenta la proporición de un 18% a un 40%

In [56]:
# Aplicar SMOTE SOLO al train
smote2 = SMOTE(random_state=42, sampling_strategy=0.4)

X_train_resampled2, y_train_resampled2 = smote2.fit_resample(
    X_train,
    y_train
)

In [57]:
print(y_train.value_counts())
print(y_train_resampled2.value_counts())

Revenue
0          8338
1          1526
Name: count, dtype: int64
Revenue
0          8338
1          3335
Name: count, dtype: int64


### Definición de metodología de validación

In [58]:
from sklearn.model_selection import StratifiedKFold

cv_strategy = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

### Creación de pipeline

In [59]:
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline
pipeline_lr = Pipeline([
    ('smote', smote),
    ('logisticRegression', LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    ))
])

### Definición de las métricas

In [28]:
#scoring = ['roc_auc','average_precision', 'f1']

In [60]:
scoring = "roc_auc"

### Aplicación de la metodología de validación

In [61]:
y_train = y_train.values.ravel()

In [62]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipeline_lr,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=scoring
)

In [63]:
scores

array([0.88293287, 0.90063635, 0.89654551, 0.90626322, 0.86759277,
       0.91557806, 0.91193361, 0.89270163, 0.91123508, 0.90402435])